# CP2K band unfolding from SCF AO data

This notebook is launched from a CP2K SCF overlap calculation in the surfaces app. It uses the diagonalization-step WFN from the remote folder and the retrieved sparse AO overlap matrix.


In [ ]:
%load_ext aiida
%aiida

import urllib.parse as urlparse
from pathlib import Path

import ipywidgets as ipw
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
from aiida import orm

from useful_notebooks_cp2k_unfolding import (
    build_modulo_lattice_ao_mapping,
    folded_kpoints_from_supercell_matrix,
    guess_2d_lattice_type,
    infer_aos_per_symbol_from_wfn,
    integer_supercell_matrix,
    kfrac_to_cart,
    mo_norms_sparse,
    plot_unfolded_kpath,
    print_eigenvalue_summary,
    project_kpoints_to_kpath,
    read_cp2k_wfn,
    read_sparse_overlap_npz,
    standard_kpath,
    spectral_weight_full,
    unfold_band_weights_full,
)


In [ ]:
pk = int(urlparse.parse_qs(urlparse.urlsplit(jupyter_notebook_url).query)["pk"][0])
workcalc = orm.load_node(pk)

if not workcalc.label.startswith("CP2K_SCF"):
    raise ValueError(f"Expected a CP2K_SCF workchain, got label {workcalc.label!r}.")
if "remote_folder" not in workcalc.outputs:
    raise ValueError("The workchain has no remote_folder output.")
if "sparse_overlap_retrieved" not in workcalc.outputs:
    raise ValueError("The workchain has no sparse_overlap_retrieved output. Run SCF with sparse AO overlap retrieval enabled.")

structure = workcalc.inputs.structure
ase_atoms = structure.get_ase()
symbols = ase_atoms.get_chemical_symbols()
coords = ase_atoms.get_positions()
supercell_vectors_3d = np.asarray(ase_atoms.cell.array, dtype=float)
remote_path = Path(workcalc.outputs.remote_folder.get_remote_path())
wfn_file = remote_path / "aiida-RESTART.wfn"

print(f"SCF workchain PK: {workcalc.pk}")
print(f"Label: {workcalc.label}")
print(f"Number of atoms: {len(symbols)}")
print(f"Remote folder: {remote_path}")
print(f"WFN file exists: {wfn_file.exists()} ({wfn_file})")
print("Supercell vectors [Angstrom]:")
print(supercell_vectors_3d)


In [ ]:
def _vector_to_text(vector):
    return " ".join(f"{x:.10g}" for x in vector)


def _matrix_to_text(matrix):
    return "\n".join(_vector_to_text(row) for row in matrix)


def _parse_vector(text):
    text = text.strip()
    if not text:
        return None
    values = [float(x) for x in text.replace(",", " " ).split()]
    if len(values) != 3:
        raise ValueError("Each primitive vector must contain three numbers.")
    return np.asarray(values, dtype=float)


def _lattice_matrix(vectors):
    vectors = np.asarray(vectors, dtype=float)
    dim = vectors.shape[0]
    return vectors[:dim, :dim].T


def _primitive_from_supercell_matrix(supercell_vectors, matrix):
    dim = matrix.shape[0]
    supercell_lattice = _lattice_matrix(supercell_vectors)
    primitive_lattice = supercell_lattice @ np.linalg.inv(matrix)
    vectors = np.zeros((dim, 3), dtype=float)
    vectors[:, :dim] = primitive_lattice.T
    return vectors


def read_primitive_vectors():
    vectors = [_parse_vector(widget.value) for widget in (a1_widget, a2_widget, a3_widget)]
    if vectors[0] is None:
        raise ValueError("a1 is required. Leave only a2/a3 empty for lower-dimensional systems.")
    if vectors[1] is None and vectors[2] is not None:
        raise ValueError("a2 cannot be empty when a3 is set.")
    return np.asarray([vector for vector in vectors if vector is not None], dtype=float)


def snap_primitive_vectors_to_supercell(approx_vectors, supercell_vectors):
    dim = approx_vectors.shape[0]
    approx_lattice = _lattice_matrix(approx_vectors)
    supercell_lattice = _lattice_matrix(supercell_vectors)
    matrix_float = np.linalg.solve(approx_lattice, supercell_lattice)
    matrix_center = np.rint(matrix_float).astype(int)

    best = None
    for delta in np.ndindex(*([3] * (dim * dim))):
        delta_matrix = np.asarray(delta, dtype=int).reshape(dim, dim) - 1
        matrix = matrix_center + delta_matrix
        det = np.linalg.det(matrix)
        if abs(det) < 0.5:
            continue
        snapped = _primitive_from_supercell_matrix(supercell_vectors, matrix)
        try:
            integer_supercell_matrix(snapped, supercell_vectors)
        except ValueError:
            continue
        correction = snapped - approx_vectors
        score = float(np.linalg.norm(correction))
        det_penalty = abs(int(round(abs(det)))) * 1e-8
        item = (score + det_penalty, score, matrix, snapped, matrix_float)
        if best is None or item[0] < best[0]:
            best = item

    if best is None:
        raise ValueError(
            "Could not snap these vectors to an exact integer tiling of the supercell. "
            "Try vectors closer to the primitive cell."
        )

    _, correction_norm, matrix, snapped, matrix_float = best
    return snapped, matrix, matrix_float, correction_norm


def update_snapped_preview(*_):
    try:
        approx_vectors = read_primitive_vectors()
        dim = approx_vectors.shape[0]
        snapped, matrix, matrix_float, correction_norm = snap_primitive_vectors_to_supercell(
            approx_vectors, supercell_vectors_3d[:dim]
        )
    except Exception as exc:
        snapped_vectors_widget.value = f"Cannot snap vectors yet: {exc}"
        snap_matrix_widget.value = ""
        snap_correction_widget.value = ""
        return

    snapped_vectors_widget.value = _matrix_to_text(snapped)
    snap_matrix_widget.value = str(matrix)
    snap_correction_widget.value = f"{correction_norm:.6g} Angstrom"


a1_widget = ipw.Text(
    value=_vector_to_text(supercell_vectors_3d[0]),
    description="a1:",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)
a2_widget = ipw.Text(
    value=_vector_to_text(supercell_vectors_3d[1]),
    description="a2:",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)
a3_widget = ipw.Text(
    value="",
    description="a3:",
    placeholder="Leave empty for 2D; leave a2 and a3 empty for 1D",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

snapped_vectors_widget = ipw.Textarea(
    value="",
    description="snapped:",
    disabled=True,
    rows=3,
    style={"description_width": "initial"},
    layout={"width": "70%"},
)
snap_matrix_widget = ipw.Textarea(
    value="",
    description="M:",
    disabled=True,
    rows=3,
    style={"description_width": "initial"},
    layout={"width": "40%"},
)
snap_correction_widget = ipw.Text(
    value="",
    description="correction:",
    disabled=True,
    style={"description_width": "initial"},
    layout={"width": "28%"},
)

for widget in (a1_widget, a2_widget, a3_widget):
    widget.observe(update_snapped_preview, names="value")

lattice_type_widget = ipw.Dropdown(
    value="auto",
    options=["auto", "1d", "square", "rectangular", "hexagonal", "oblique"],
    description="Path type:",
    style={"description_width": "initial"},
)
spin_widget = ipw.BoundedIntText(
    value=0,
    min=0,
    max=1,
    description="Spin index:",
    style={"description_width": "initial"},
)
emin_widget = ipw.FloatText(
    value=-3.0,
    description="E min [eV]:",
    style={"description_width": "initial"},
)
emax_widget = ipw.FloatText(
    value=3.0,
    description="E max [eV]:",
    style={"description_width": "initial"},
)
run_button = ipw.Button(description="Run unfolding", button_style="primary")
output = ipw.Output()

update_snapped_preview()

display(
    ipw.VBox([
        ipw.HTML("<b>Approximate primitive vectors in Angstrom</b>"),
        a1_widget,
        a2_widget,
        a3_widget,
        ipw.HTML("<b>Exact primitive vectors used for unfolding</b>"),
        snapped_vectors_widget,
        ipw.HBox([snap_matrix_widget, snap_correction_widget]),
        ipw.HBox([lattice_type_widget, spin_widget, emin_widget, emax_widget]),
        run_button,
        output,
    ])
)



In [ ]:
def load_sparse_overlap_from_workchain(workcalc):
    with workcalc.outputs.sparse_overlap_retrieved.open("sparse_overlap.npz", "rb") as handle:
        parsed = read_sparse_overlap_npz(handle)
    return parsed.matrix, parsed


def run_unfolding(_=None):
    with output:
        clear_output()
        approximate_vectors = read_primitive_vectors()
        dim = approximate_vectors.shape[0]
        supercell_vectors = supercell_vectors_3d[:dim]
        primitive_vectors, M_check, M_float, correction_norm = snap_primitive_vectors_to_supercell(
            approximate_vectors, supercell_vectors
        )
        update_snapped_preview()

        lattice_type = lattice_type_widget.value
        if lattice_type == "auto":
            lattice_type = guess_2d_lattice_type(primitive_vectors) if dim == 2 else "1d"

        print("Dimensionality:", dim)
        print("Approximate primitive vectors [Angstrom]:")
        print(approximate_vectors)
        print("Exact primitive vectors used [Angstrom]:")
        print(primitive_vectors)
        print("Correction norm [Angstrom]:", correction_norm)
        print("Supercell vectors used [Angstrom]:")
        print(supercell_vectors)
        print("Lattice/path type:", lattice_type)

        print("Estimated supercell matrix from approximate vectors:")
        print(M_float)
        print("Integer supercell matrix M, defined by S = A @ M:")
        print(M_check)
        print("det(M):", int(round(abs(np.linalg.det(M_check)))))

        print("\nReading WFN...")
        wfn = read_cp2k_wfn(
            wfn_file,
            emin=emin_widget.value,
            emax=emax_widget.value,
        )
        spin = spin_widget.value
        C = wfn.coeffs[spin]
        print("selected eigenvalues:", wfn.evals_ev[spin].shape)
        print("coefficient array:", C.shape)
        print_eigenvalue_summary(wfn, spin=spin, n=8)

        print("\nReading sparse overlap NPZ...")
        S, overlap_data = load_sparse_overlap_from_workchain(workcalc)
        if S.shape != (C.shape[1], C.shape[1]):
            raise ValueError(f"Overlap shape {S.shape} does not match WFN AO count {C.shape[1]}.")
        print("S shape:", S.shape)
        print("S nnz:", S.nnz)
        print("first 10 diagonal elements:", S.diagonal()[:10])
        print("max asymmetry:", abs(S - S.T).max())

        norms_S = mo_norms_sparse(C, S)
        print("MO norms C S C:")
        print(norms_S)

        aos_per_symbol = infer_aos_per_symbol_from_wfn(symbols, C.shape[1])
        print("AO count per symbol inferred from WFN:", aos_per_symbol)

        mapping = build_modulo_lattice_ao_mapping(
            symbols=symbols,
            coords_cart=coords,
            primitive_vectors=primitive_vectors,
            supercell_vectors=supercell_vectors,
            aos_per_symbol=aos_per_symbol,
            tol=1e-5,
        )
        print("nao supercell:", mapping.nao_super)
        print("nao primitive:", mapping.nao_prim)
        print("number of primitive replicas:", mapping.nrep)
        print("atom -> primitive basis atom:")
        print(mapping.atom_to_basis)
        print("atom -> replica:")
        print(mapping.atom_to_replica)

        k_gamma = np.array([0.0, 0.0, 0.0])
        weights_gamma = np.array([
            spectral_weight_full(C[imo], k_gamma, S, mapping)
            for imo in range(C.shape[0])
        ])
        print("full-projector Gamma weights:")
        print(weights_gamma)

        k_frac_folded = folded_kpoints_from_supercell_matrix(mapping.supercell_integer_matrix)
        k_cart_folded = kfrac_to_cart(k_frac_folded, primitive_vectors)
        weights_folded = unfold_band_weights_full(C, k_cart_folded, S, mapping)
        print("weights_folded shape:", weights_folded.shape)
        print("sum over folded k-points per MO:")
        print(weights_folded.sum(axis=0))

        hs_points, hs_path = standard_kpath(dim, lattice_type=lattice_type, primitive_vectors=primitive_vectors)
        path_k_indices, path_x, path_segments, path_t, path_q_equiv, x_ticks = project_kpoints_to_kpath(
            k_frac_folded,
            hs_points,
            hs_path,
            primitive_vectors,
            tol_cart=1e-6,
        )
        print("folded k-points on this path:", path_k_indices)
        print("number of folded k-points on this path:", len(path_k_indices), "of", len(k_frac_folded))
        if len(path_k_indices) < len(k_frac_folded):
            print(
                "Only k-points exactly on the selected 1D path are plotted; "
                "the remaining folded k-points live elsewhere in the 2D Brillouin zone."
            )
        print("path:", "-".join(hs_path))

        if dim == 2:
            all_k_frac = k_frac_folded % 1.0
            path_k_frac = path_q_equiv % 1.0 if len(path_q_equiv) else np.empty((0, 2))
            path_nodes = np.asarray([hs_points[label] for label in hs_path])
            fig, ax_k = plt.subplots(figsize=(4.5, 4.0))
            ax_k.scatter(all_k_frac[:, 0], all_k_frac[:, 1], s=18, alpha=0.45, label="folded k-points")
            if len(path_k_frac):
                ax_k.scatter(path_k_frac[:, 0], path_k_frac[:, 1], s=42, color="tab:red", label="on plotted path")
            ax_k.plot(path_nodes[:, 0], path_nodes[:, 1], color="black", linewidth=1.2, label="selected path")
            for label, point in hs_points.items():
                ax_k.text(point[0], point[1], label, ha="center", va="bottom")
            ax_k.set_xlabel("primitive reciprocal fractional k1")
            ax_k.set_ylabel("primitive reciprocal fractional k2")
            ax_k.set_xlim(-0.03, 1.03)
            ax_k.set_ylim(-0.03, 1.03)
            ax_k.set_aspect("equal", adjustable="box")
            ax_k.legend(loc="upper right", fontsize=8)
            ax_k.figure.tight_layout()
            plt.show()

        energies_shifted = wfn.evals_ev[spin] - wfn.ref_energy_ev
        ax = plot_unfolded_kpath(
            path_k_indices=path_k_indices,
            path_x=path_x,
            x_ticks=x_ticks,
            x_tick_labels=hs_path,
            energies_ev=energies_shifted,
            weights=weights_folded,
        )
        plt.show()


run_button.on_click(run_unfolding)


